# Формирование датасета, EDA и очистка

## Импортируем библиотеки и создаем полезные функции

In [86]:
import pandas as pd
from dateutil.relativedelta import relativedelta

In [87]:
# Функция для замены значения на ключ словаря
def replace_with_dict_key(value, rep_dict):
    for key, values_list in rep_dict.items():
        if value in values_list:
            return key
    return None 

## Загружаем данные

In [88]:
raw_df = pd.read_csv(r'data/undefind_DSMED.csv', sep=';')
raw_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 36297 entries, 0 to 36296
Data columns (total 13 columns):
 #   Column                                Non-Null Count  Dtype  
---  ------                                --------------  -----  
 0   ID истории болезни                    36297 non-null  object 
 1   Осн. диаг. при выписке МКБ10 (текст)  36297 non-null  object 
 2   Заголовок документа                   36297 non-null  object 
 3   Кол. лаб. показатель                  36236 non-null  object 
 4   Значение кол. показателя              36236 non-null  float64
 5   Ед. изм. кол. показателя              36236 non-null  object 
 6   Норма кол. показателя                 36236 non-null  object 
 7   Флаг нормы кол. показателя            36236 non-null  object 
 8   Кач. лаб. показатель                  7005 non-null   object 
 9   Значение кач. показателя              7005 non-null   object 
 10  Норма кач. показателя                 7005 non-null   object 
 11  Пол            

## Собственно, формирование датасета

In [89]:
# Посмотрим количество уникальных значений по колонкам
raw_df.nunique()

ID истории болезни                       243
Осн. диаг. при выписке МКБ10 (текст)      11
Заголовок документа                        1
Кол. лаб. показатель                     131
Значение кол. показателя                2608
Ед. изм. кол. показателя                  11
Норма кол. показателя                    123
Флаг нормы кол. показателя                 4
Кач. лаб. показатель                       7
Значение кач. показателя                 155
Норма кач. показателя                      5
Пол                                        2
Дата рождения пациента                   104
dtype: int64

Так как прогнозировать будем по показателям, то наши признаки в эталонном датасете - показатели лабораторных тестов

In [90]:
# Посмотрим состав Кол. лабораторных показателей
raw_df['Кол. лаб. показатель'].value_counts()

Кол. лаб. показатель
Гемоглобин (HGB)                                        1114
Средний объем эритроцита (MCV)                          1114
Гематокрит (HCT)                                        1114
Среднее содержание гемоглобина в эритроците (MCH)       1114
Средняя концентрация гемоглобина в эритроците (MCHC)    1114
                                                        ... 
Тромбокрит                                                 1
Эозинофилы, абсолютное количество                          1
Базофилы, абсолютное количество                            1
СОЭ по Панченкову                                          1
MXD#                                                       1
Name: count, Length: 131, dtype: int64

In [91]:
replacement_dict = {
    # Красная кровь
    'rbc': ['Эритроциты', 'Эритроциты (RBC)', 'Общее количество эритроцитов (RBC)', 'RBC'],
    'hgb': ['Гемоглобин', 'Гемоглобин (HGB)', 'HGB'],
    'hct': ['Гематокрит', 'Гематокрит (HCT)', 'HCT'],
    'mcv': ['Средний объем эритроцита', 'Средний объем эритроцита (MCV)', 'MCV'],
    'mch': ['Среднее содержание гемоглобина в эритроците', 'Среднее содержание гемоглобина в эритроците (MCH)', 'MCH'],
    'mchc': ['Средняя концентрация гемоглобина в эритроците', 'Средняя концентрация гемоглобина в эритроците (MCHC)', 'MCHC'],
    'rdw': ['Ширина распределения эритроцитов', 'Ширина распределения эритроцитов по объему (RDW)', 'RDW', 'Ширина распределения эритроцитов (RDW)'],
    'rdv_cv': ['Ширина распределения эритроцитов по объему, коэффициент вариации (RDW-CV)', 'Ширина распределения эритроцитов по объему, коэффициент вариации', 'RDW-CV '],
    'rdv_sd': ['Ширина распределения эритроцитов по объему, стандартное отклонение (RDW-SD)','Ширина распределения эритроцитов по объему, стандартное отклонение','Ширина распределения эритроцитов, стандартное отклонение (RDW-SD)', 'RDW-SD'],
    'ret_rel': ['RET%', 'Ретикулоциты %'],
    'ret_abs': ['Ретикулоциты кол-во'],    
    'cp': ['Цветовой показатель'],

    # Белая кровь
    'wbc': ['Лейкоциты', 'Лейкоциты (WBC)', 'Общее количество лейкоцитов (WBC)', 'WBC'],

    # Лейкоформула
    'ne_abs': ['Нейтрофилы, абсолютное количество', 'Нейтрофилы, абсолютное количество (NE#)', 'Абсолютное количество нейтрофилов (NE#)', 'Нейтрофилы #'],
    'ne_rel': ['Нейтрофилы, относительное количество', 'Нейтрофилы, относительное количество (NE%)', 'Относительное количество нейтрофилов (NE%)', 'Нейтрофилы %', 'NE%'],
    'ly_abs': ['Лимфоциты, абсолютное количество', 'Лимфоциты, абсолютное количество (LY#)', 'Абсолютное количество лимфоцитов (LY#)', 'Лимфоциты #'],
    'ly_rel': ['Лимфоциты, относительное количество', 'Лимфоциты, относительное количество (LY%)', 'Лимфоциты %', 'Относительное количество лимфоцитов (LY%)', 'Лимфоциты %', 'LY%', 'Лимфоциты'],
    'mo_abs': ['Моноциты, абсолютное количество', 'Моноциты, абсолютное количество (MO#)', 'Абсолютное количество моноцитов (MO#)', 'Моноциты #'],
    'mo_rel': ['Моноциты, относительное количество', 'Моноциты, относительное количество (MO%)', 'Моноциты %', 'Моноциты %', 'Относительное количество моноцитов (MO%)', 'MO%', 'Моноциты'],
    'eo_abs': ['Эозинофилы, абсолютное количество', 'Эозинофилы, абсолютное количество (EO#)', 'Абсолютное количество эозинофилов (EO#)', 'Эозинофилы # '],
    'eo_rel': ['Эозинофилы, относительное количество', 'Эозинофилы, относительное количество (EO%)', 'Эозинофилы %', 'Относительное количество эозинофилов (EO%)', 'EO%', 'Эозинофилы'],
    'ba_abs': ['Базофилы, абсолютное количество', 'Базофилы, абсолютное количество (BA#)', 'Абсолютное количество базофилов (BA#)', 'Базофилы #', 'Базофилы # '],
    'ba_rel': ['Базофилы, относительное количество', 'Базофилы, относительное количество (BA%)', 'Базофилы %', 'Относительное количество базофилов (BA%)', 'BA%', 'Базофилы'],
    'pal': ['Палочкоядерные'],
    'seg': ['Сегментоядерные'],
    'mxd_abs': ['Смешанная фракция, абсолютное количество (MXD#)', 'MXD# ', 'Смешанная фракция, абсолютное количество', 'MXD#'],
    'mxd_rel': ['Смешанная фракция, относительное количество (MXD%)', 'MXD%', 'Смешанная фракция, относительное количество', 'MXD'],
    'plasma': ['Плазматические клетки', 'Плазматич. клетки'], # в норме - 0-1%

    # Тромбоциты
    'plt': ['Тромбоциты', 'Тромбоциты (PLT)', 'PLT'],
    'pct': ['Тромбокрит', 'Тромбокрит (PCT)', 'PCT'],
    'mpv': ['Средний объем тромбоцита', 'Средний объём тромбоцитов', 'Средний объем тромбоцита (MPV)', 'MPV (Средний объём тромбоцитов)', 'MPV ', 'MPV'],
    'pdw': ['Ширина распределения тромбоцитов', 'Ширина распределения тромбоцитов по объему', 'Ширина распределения тромбоцитов (PDW)', 'PDW'],
    'plcr': ['P-LCR'],

    'soe': ['СОЭ Вест.', 'Скорость оседания эритроцитов (СОЭ) по Вестергрену', 'СОЭ по Панченкову', 'СОЭ Панч.'],

    'myelo': ['Миелоциты'], # в норме - 0 появление в периферической крови всегда указывает на патологию
    'yunye': ['Юные'], # в норме - 0 всегда требует обследования
    'blasty': ['Бласты'], # в норме - 0 опухолевые клетки, лейкоз
    'normobl_abs': ['Нормобласты', 'Нормобласты #'], # в норме - 0
    'normobl_rel': ['Нормобласты %'], # в норме - 0
    'prolym': ['Пролимфоциты'], # в норме - 0 "красный флаг", требующий исключения злокачественных заболеваний крови!
    'promyelo': ['Промиелоциты'], # в норме - 0  тревожный сигнал, требующий исключения лейкозов и других тяжелых заболеваний крови

    'noclass_abs': ['Неклассифицируемые кол-во'],
    'noclass_rel': ['Неклассифицируемые %'],
}

В составе тестов увидели много синонимов - нужно мапить на уникальные признаки. Сформировали словарь для маппинга

In [92]:
# начинаем первую очистку и создание нужных признаков
prep_df = raw_df.copy()
# Удаляем строки, где вообще нет показателей
# Конечно мы потеряем 61 строку с комментариями, но они либо о браке, либо непонятные
prep_df = prep_df.dropna(subset=['Кол. лаб. показатель'])
# Посчитаем возраст пациента в годах на 21.07.2025 - начало хакатона. Других дат нет
age_date = pd.to_datetime('2025-07-21')
prep_df['Дата рождения пациента'] = pd.to_datetime(prep_df['Дата рождения пациента'])
prep_df['age'] = prep_df['Дата рождения пациента'].apply(lambda x: relativedelta(age_date, x).years)
# Закодируем пол числом
prep_df['gender'] = prep_df['Пол'].apply(lambda x: 0 if 'ж' in x.lower() else 1)


In [93]:
# Формируем номер лаб исследования исходя из логики, что тесты в датасете расположены
# по хронологии сверху вниз и в одном ЛИ тесты не повторяются
lab_study = []
study_prefix = 0

def create_lab_study_num(lab_test):

    global lab_study
    global study_prefix
    res = 'result_'

    if lab_test in lab_study:
        lab_study.clear()
        study_prefix += 1
    else:
        lab_study.append(lab_test)
    return res + str(study_prefix)
prep_df['study_num'] = prep_df['Кол. лаб. показатель'].apply(lambda x: create_lab_study_num(x))
# формируем уникальный ID ЛИ из ID истории и номера ЛИ
prep_df['study_ID'] = prep_df['ID истории болезни'] + '_' + prep_df['study_num']

In [94]:
# добавляем унифицированные коды тестов из словаря синонимов
prep_df['lab_test'] = prep_df['Кол. лаб. показатель'].apply(lambda x: replace_with_dict_key(x, replacement_dict))

In [95]:
# Переводим г/дл в г/л в MCHC
prep_df.loc[prep_df['Ед. изм. кол. показателя'] == 'г/дл', 'Значение кол. показателя'] *= 10
prep_df.loc[prep_df['Ед. изм. кол. показателя'] == 'г/дл', 'Ед. изм. кол. показателя'] = 'г/л'

In [96]:
# посмотрим сколько тестов осталось после унификации
prep_df['lab_test'].value_counts()

lab_test
ly_rel         2213
hgb            1619
rbc            1619
mchc           1619
hct            1619
mch            1619
mcv            1619
plt            1617
wbc            1616
mo_rel         1549
eo_rel         1473
ly_abs         1470
ne_rel         1332
ne_abs         1319
ba_rel         1177
mpv             982
pct             925
cp              832
rdw             829
soe             794
mo_abs          784
seg             740
eo_abs          739
pal             736
ba_abs          732
rdv_sd          698
rdv_cv          696
pdw             657
mxd_abs         517
mxd_rel         517
myelo           316
yunye           304
noclass_rel     239
normobl_abs     192
blasty          143
noclass_abs     117
plcr             81
ret_rel          57
ret_abs          44
plasma           30
promyelo         21
normobl_rel      18
prolym           16
Name: count, dtype: int64

In [97]:
# Сформируем справочный фрейм с данными пациенов, чтоб домержить в основной датасет
pasp_df = prep_df[['study_ID', 'gender', 'age']].drop_duplicates().copy()
pasp_df

,study_ID,gender,age
0,2e1d0b3f-488a-11ed-ab5a-0050568844e6_result_0,0,62
16,2e1d0b3f-488a-11ed-ab5a-0050568844e6_result_1,0,62
47,2e1d0b3f-488a-11ed-ab5a-0050568844e6_result_2,0,62
78,2e1d0b3f-488a-11ed-ab5a-0050568844e6_result_3,0,62
107,2e1d0b3f-488a-11ed-ab5a-0050568844e6_result_4,0,62
...,...,...,...
36176,6166e156-c1af-11ed-8602-005056880ecb_result_2637,1,43
36203,6166e156-c1af-11ed-8602-005056880ecb_result_2638,1,43
36227,6166e156-c1af-11ed-8602-005056880ecb_result_2639,1,43
36253,6166e156-c1af-11ed-8602-005056880ecb_result_2640,1,43


In [98]:
# Проверим референсные значения
ref_df = prep_df[['Пол', 'lab_test', 'Норма кол. показателя']].drop_duplicates()
# ref_df = prep_df[['Пол', 'Кол. лаб. показатель', 'Норма кол. показателя']].drop_duplicates()

ref_df[['lo_ref', 'hi_ref']] = ref_df['Норма кол. показателя'].str.split(':', n=1, expand=True)
ref_df.drop(columns=['Норма кол. показателя'], inplace=True)
ref_df['lo_ref'] = ref_df['lo_ref'].str.replace(',', '.')
ref_df['hi_ref'] = ref_df['hi_ref'].str.replace(',', '.')
ref_df.dropna(inplace=True)
ref_df[['lo_ref', 'hi_ref']] = ref_df[['lo_ref', 'hi_ref']].astype('float')
ref_df.drop_duplicates(inplace=True)
ref_df.sort_values(['Пол', 'lab_test'])
# ref_df.sort_values(['Пол', 'Кол. лаб. показатель'])


,Пол,lab_test,lo_ref,hi_ref
37,Ж,ba_abs,0.00,0.10
32,Ж,ba_rel,0.00,2.00
308,Ж,blasty,0.00,0.00
38,Ж,cp,0.80,1.05
1621,Ж,cp,0.85,1.15
...,...,...,...,...
484,М,soe,2.00,20.00
3590,М,soe,1.00,20.00
462,М,wbc,4.00,11.00
11344,М,wbc,4.00,8.80


Видим, что референсные значения разные для одних и тех же тестов, даже для одного пола. Похоже, на них не сможем ориентироваться, будем ориентироваться на флаги результата, т.к. они индивидуальны для каждого теста.

In [99]:
# Закодируем флаг результата числом: норма = 0, повышенный = 1, пониженный = -1
# пригодится, если захотим обучать не на абсолютах, а по флагам результата
prep_df['result'] = prep_df['Флаг нормы кол. показателя'].apply(lambda x: 0 if 'норм' in x.lower()
                                                                else (-1 if "пониж" in x.lower()
                                                                      else (1 if "повыш" in x.lower()
                                                                            else x)))

Займемся качественными показателями. У них нет числового результата и результатом будем считать флаг наличия или отсутствич признака.
Часть качественных признаков названа явно, а часть их описаны в комментариях, не все из которых полезны.

In [100]:
# Выделяем кач показатели, которые не в комментариях
mask = prep_df['Кач. лаб. показатель'].notna() & prep_df['Кач. лаб. показатель'].ne('Комментарий')
real_qual_df = prep_df[mask][['study_ID', 'Кач. лаб. показатель', 'Значение кач. показателя', 'Норма кач. показателя']].copy()
real_qual_df

,study_ID,Кач. лаб. показатель,Значение кач. показателя,Норма кач. показателя
930,76416b20-b70b-11ec-ab54-0050568844e6_result_38,Анизоцитоз,+,+-
931,76416b20-b70b-11ec-ab54-0050568844e6_result_39,Гипохромия,+,+-
932,76416b20-b70b-11ec-ab54-0050568844e6_result_39,Анизоцитоз,+,+-
933,76416b20-b70b-11ec-ab54-0050568844e6_result_40,Гипохромия,+,+-
934,76416b20-b70b-11ec-ab54-0050568844e6_result_40,Анизоцитоз,+,+-
...,...,...,...,...
36037,20f82885-b5cf-11ee-ab6f-0050568844e6_result_2627,Пойкилоцитоз,+++,+-
36039,20f82885-b5cf-11ee-ab6f-0050568844e6_result_2627,Анизоцитоз,+++,+-
36040,20f82885-b5cf-11ee-ab6f-0050568844e6_result_2628,Пойкилоцитоз,+++,+-
36042,20f82885-b5cf-11ee-ab6f-0050568844e6_result_2628,Анизоцитоз,+++,+-


In [101]:
# Посмотрим на значения нормы
real_qual_df['Норма кач. показателя'].value_counts(dropna=False)

Норма кач. показателя
+-             2170
Общая норма     223
0:1              29
0:2              20
Name: count, dtype: int64

In [102]:
# Посмотрим варианты нормы для разных кач показателей
real_qual_df.groupby(['Кач. лаб. показатель', 'Норма кач. показателя']).count()

study_ID  Значение кач. показателя
Кач. лаб. показатель Норма кач. показателя                                    
Анизоцитоз           +-                          910                       910
                     Общая норма                 113                       113
Гипохромия           +-                          191                       191
                     Общая норма                  54                        54
Макроцитоз           +-                          141                       141
Микроцитоз           +-                          139                       139
                     Общая норма                  29                        29
Нормобласты          0:1                          29                        29
                     0:2                          20                        20
Пойкилоцитоз         +-                          789                       789
                     Общая норма                  27                        27

In [103]:
# Посмотрим на реальные значения кач показателей
real_qual_df['Значение кач. показателя'].value_counts(dropna=False)

Значение кач. показателя
+            1250
++            895
+++           182
+-             66
3.0:100.0      29
3:100          20
Name: count, dtype: int64

In [104]:
# Сформируем словарь для кодирования результатов
real_result_dict = {
    0: ['+', '-', '+-'],
    1: ['++', '+++', '3.0:100.0', '3:100']
}
# Сформируем словарь для перекодирования наименований показателей (задалбывает переключать раскладку)
rus_to_eng_dict = {
    'anisocytos': ['Анизоцитоз'],
    'hypochromia': ['Гипохромия'],
    'macrocytos': ['Макроцитоз'],
    'microcytos': ['Микроцитоз'],
    'poikilocytos': ['Пойкилоцитоз'],
    'normoblast': ['Нормобласты']
}
# Перекодируем результаты числами
real_qual_df['Значение кач. показателя'] = real_qual_df['Значение кач. показателя'].apply(lambda x: replace_with_dict_key(x, real_result_dict))
real_qual_df['Кач. лаб. показатель'] = real_qual_df['Кач. лаб. показатель'].apply(lambda x: replace_with_dict_key(x, rus_to_eng_dict))
real_qual_df.drop(columns=['Норма кач. показателя'], inplace=True)
real_qual_df.drop_duplicates()

,study_ID,Кач. лаб. показатель,Значение кач. показателя
930,76416b20-b70b-11ec-ab54-0050568844e6_result_38,anisocytos,0
931,76416b20-b70b-11ec-ab54-0050568844e6_result_39,hypochromia,0
932,76416b20-b70b-11ec-ab54-0050568844e6_result_39,anisocytos,0
933,76416b20-b70b-11ec-ab54-0050568844e6_result_40,hypochromia,0
934,76416b20-b70b-11ec-ab54-0050568844e6_result_40,anisocytos,0
...,...,...,...
36037,20f82885-b5cf-11ee-ab6f-0050568844e6_result_2627,poikilocytos,1
36039,20f82885-b5cf-11ee-ab6f-0050568844e6_result_2627,anisocytos,1
36040,20f82885-b5cf-11ee-ab6f-0050568844e6_result_2628,poikilocytos,1
36042,20f82885-b5cf-11ee-ab6f-0050568844e6_result_2628,anisocytos,1


In [105]:
# переведем значения кач показателей из строк в столбцы
work_real_qual_df = real_qual_df.pivot_table(index='study_ID', columns='Кач. лаб. показатель', values='Значение кач. показателя', aggfunc='first')

Предлагаю комментарии в признаки не добавлять. Работы куча, а толку - ноль.

In [106]:
# # Выделяем кач показатели, которые в комментариях
# mask = prep_df['Кач. лаб. показатель'] == 'Комментарий'
# real_comm_df = prep_df[mask][['study_ID', 'Значение кач. показателя']].copy()
# real_comm_df

In [107]:
# real_comm_df = real_comm_df[~real_comm_df['Значение кач. показателя'].str.contains('проверен', na=False)]
# real_comm_df = real_comm_df[~real_comm_df['Значение кач. показателя'].str.contains('Проверен', na=False)]
# real_comm_df = real_comm_df[~real_comm_df['Значение кач. показателя'].str.contains('ПРОВЕРЕН', na=False)]
# real_comm_df = real_comm_df[~real_comm_df['Значение кач. показателя'].str.contains('тромб.пров', na=False)]
# real_comm_df = real_comm_df[~real_comm_df['Значение кач. показателя'].str.contains('тромбоциты по мазку', na=False)]
# real_comm_df = real_comm_df[~real_comm_df['Значение кач. показателя'].str.contains('тромбоциты соответствуют', na=False)]
# real_comm_df = real_comm_df[~real_comm_df['Значение кач. показателя'].str.contains('невозм', na=False)]

In [108]:
# real_comm_df[real_comm_df['Значение кач. показателя'].str.contains('промоноцит', na=False)] = 'promonocit'
# real_comm_df[real_comm_df['Значение кач. показателя'].str.contains('полихроматофильные эритроциты', na=False)] = 'polychrome_rbc'

In [109]:
# real_comm_df['Значение кач. показателя'].value_counts()

In [110]:
# Соберем датасет для работы
work_df = prep_df.pivot_table(values='Значение кол. показателя', columns='lab_test', index='study_ID', aggfunc='first')
work_df = work_df.merge(pasp_df, on='study_ID', how='left')
work_df = work_df.merge(work_real_qual_df, on='study_ID', how='left')

In [111]:
# Это код для формирования датасета для обучения на основе флагов результатов
flag_work_df = prep_df.pivot_table(values='result', columns='lab_test', index='study_ID', aggfunc='first')
flag_work_df = flag_work_df.merge(pasp_df, on='study_ID', how='left')
flag_work_df = flag_work_df.merge(work_real_qual_df, on='study_ID', how='left')

In [ ]:
# # Выгрузим датасеты в файл
# work_df.to_csv('data/work_df.csv', index=False)
# flag_work_df.to_csv('data/flag_work_df.csv', index=False)